In [3]:
# =============================================================================
# C-PINN-ATT  --  COMPLETE SINGLE-CELL IMPLEMENTATION
# Cascaded Physics-Informed Neural Network with Multi-Head Attention
# for the Sill-Controlled COMPACT Hydraulic Jump
# -----------------------------------------------------------------------------
# Consolidated from the final authoritative reproducibility package.
#   Source: cpinn_att_reproduce.py   (core model, constraints, experiments)
#   Source: benchmark_models.py      (final benchmark: ANFIS/XGBoost/RF/MLP)
#
# SCIENTIFIC CONTENT IS UNCHANGED. This file is a code consolidation only.
#
# Architecture (parallel-cascaded, NOT a Y -> S -> X chain):
#   M1 : inputs (Fr1, h1_n, gamma)                  -> Y
#   M2 : inputs (Fr1, h1_n, gamma)                  -> S     (M2 never sees Y)
#   M3 : inputs (Fr1, h1_n, gamma, Yhat, Shat)      -> X     (hard routing)
#
# Constraints implemented: C1, C2, C3, C4, C5, C6, C7
# PVI: momentum feasibility from measured Fr1 and predicted Y only.
# gamma = Lr*/x contains sill-position information; X therefore remains a
# compactness-conditioned reconstruction/consistency quantity.
# -----------------------------------------------------------------------------
# ADDITIVE BLOCKS (this version only, on top of the authoritative package):
#   Section 24 : UNCERTAINTY QUANTIFICATION  -- deep ensemble + MC-Dropout,
#                combined through the law of total variance.
#   Section 25 : APPLICABILITY DOMAIN        -- PCA(2) convex hull of the
#                training inputs; test points classified as interpolation
#                (inside) vs. extrapolation (outside); relative-error (%)
#                histograms and mean/std/max statistics split by domain.
# Both blocks reuse run_cpinn()/CPINNAtt() exactly as defined below and do
# not alter the loss, the constraints, the architecture, or any reported
# table. Every number they compute is written to <OUTPUT_DIR>/csv/*.csv and
# every plot to <OUTPUT_DIR>/figures/*.png (see Section 26, MAIN EXECUTION).
# =============================================================================

# -----------------------------------------------------------------------------
# 1. IMPORTS
# -----------------------------------------------------------------------------
from __future__ import annotations

import json
import os
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats
from scipy.spatial import ConvexHull, Delaunay
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# 2. REPRODUCIBILITY / SEEDS
# -----------------------------------------------------------------------------
SEED = 42
FEATS = ["Fr1", "h1_n", "gamma"]
OUTS = ["Y", "S", "X"]
GORDER = ["\u03b3\u22481.00", "\u03b3\u22481.15", "\u03b3\u22481.30",
          "\u03b3\u22481.45", "\u03b3\u22481.60", "\u03b3\u22481.93"]
GAMMA_TH = 0.70          # compacity threshold in w(gamma), physical origin
SEEDS = (42, 0, 1, 7, 123)   # five-seed benchmark protocol

torch.manual_seed(SEED)
np.random.seed(SEED)

# -----------------------------------------------------------------------------
# 3. USER CONFIGURATION AND PATHS
#    Edit these two lines only. Relative paths keep the notebook portable.
# -----------------------------------------------------------------------------
DATA_FILE = "C:/Users/HP/Documents/FOUZI-2026/CPINN_Att_Data_Transformed_CORRECTED_AUDIT.xlsx"   # corrected dataset
OUTPUT_DIR = "C:/Users/HP/Documents/FOUZI-2026/C-PINN-ATT-FINAL-12-09-2026"                                        # results / figures

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Number of training epochs. Use 2000 for the reported results; a smaller
# value only speeds up a smoke test and will not reproduce the manuscript.
N_EPOCHS = 2000

# Which stages to execute in the main section at the bottom of this cell.
RUN_TABLE4 = True     # stratified 5-fold CV (DNN / gamma-free / C-PINN-ATT)
RUN_TABLE5 = True     # per-gamma-class cross-validated performance
RUN_TABLE6 = True     # progressive architectural ablation (fixed split)
RUN_TABLE8 = True     # fixed-split metrics, bootstrap CIs, diagnostics
RUN_BENCHMARK = True  # final five-seed benchmark (ANFIS/XGBoost/RF/MLP)
RUN_FIGURES = True    # regenerate figures belonging to the package
RUN_UQ = True         # deep-ensemble + MC-Dropout UQ, PCA(2) applicability domain

# Deep-ensemble size and MC-Dropout sample count for RUN_UQ. n_ensemble full
# models are trained from scratch, so wall-clock time is roughly
# n_ensemble x (one run_cpinn() call). Lower these for a quick smoke test.
UQ_N_ENSEMBLE = 5
UQ_MC_T = 200


# -----------------------------------------------------------------------------
# 4. DATA LOADING
# 5. PREPROCESSING / DIMENSIONLESS VARIABLES
# 6. NORMALIZATION HELPERS
# Source: cpinn_att_reproduce.py
# -----------------------------------------------------------------------------
def load_dataframe(xlsx_path: str) -> pd.DataFrame:
    """Read the 'Adimensional_A' sheet: Fr1,h1_n,gamma,Y,S,X,gamma_class,Split."""
    xl = pd.ExcelFile(xlsx_path)
    sheet = "Adimensional_A" if "Adimensional_A" in xl.sheet_names else xl.sheet_names[0]
    df = xl.parse(sheet).reset_index(drop=True)
    missing = set(FEATS + OUTS + ["gamma_class", "Split"]) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns {missing} in sheet '{sheet}'")
    assert_dimensionless(df)
    return df


def assert_dimensionless(df: pd.DataFrame, verbose: bool = True) -> None:
    """Guard: every quantity reaching the network must be dimensionless and O(1).

    The model is trained exclusively on similarity numbers. A dimensional
    quantity entering the feature set would break both the physics residuals
    (which mix Fr1, Y, S and X in single expressions) and the transfer of the
    trained model to any other channel width or depth scale. This check runs
    before training and fails loudly rather than silently degrading results.
    """
    # 1. no dimensional column may appear among features or targets
    DIM_SUFFIX = ("_cm", "_m", "_mm", "_ls", "_m3s", "_s")
    dimensional = [c for c in FEATS + OUTS if str(c).endswith(DIM_SUFFIX)]
    assert not dimensional, (
        f"Dimensional quantities in the feature/target set: {dimensional}. "
        f"Convert to ratios (X = x/h1, S = s/h1, Y = h2/h1) before training.")

    # 2. every tensor must live on an O(1)-to-O(10) scale
    # Bounds are set from the physics of the compact regime, not from this
    # particular sample, but tightly enough that a dimensional quantity
    # (x in cm is typically 10-100x its dimensionless counterpart) trips them.
    BOUNDS = {"Fr1": (1.0, 25.0), "h1_n": (0.05, 10.0), "gamma": (0.1, 5.0),
              "Y": (1.0, 30.0), "S": (0.0, 20.0), "X": (1.0, 100.0)}
    for col, (lo, hi) in BOUNDS.items():
        if col not in df.columns:
            continue
        v = df[col].to_numpy(dtype=float)
        assert np.isfinite(v).all(), f"{col} contains NaN or inf"
        assert v.min() >= lo and v.max() <= hi, (
            f"{col} outside its dimensionless range [{lo}, {hi}]: "
            f"observed [{v.min():.3f}, {v.max():.3f}]. "
            f"This usually means a dimensional column was passed by mistake.")

    # 3. scale consistency: X, S and Y are all normalised by the SAME h1, so
    #    their ratios must stay physically plausible. A uniform rescaling of one
    #    column (the classic symptom of a forgotten division by h1) breaks this.
    if {"Y", "S", "X"}.issubset(df.columns):
        ratio = (df["X"] / df["Y"]).to_numpy(dtype=float)
        assert 0.5 <= np.median(ratio) <= 12.0, (
            f"median X/Y = {np.median(ratio):.2f} is outside the plausible range "
            f"[0.5, 12] for a sill-controlled jump. One of the two columns is "
            f"probably still dimensional.")

    # 4. physical admissibility of the targets themselves
    if {"Y", "S"}.issubset(df.columns):
        bad = int(((df["S"] <= 0) | (df["S"] >= df["Y"])).sum())
        assert bad == 0, f"{bad} runs violate 0 < S < Y (constraint C5)"

    if verbose:
        print("  [check] all inputs and outputs are dimensionless and O(1)-scaled")


def table1(df: pd.DataFrame, raw_h1: np.ndarray | None = None) -> pd.DataFrame:
    """Table 1: descriptive statistics of inputs and outputs."""
    rows = []
    spec = [("Fr1", "Input"), ("h1_cm", "Input"), ("gamma", "Input"),
            ("Y", "Output"), ("S", "Output"), ("X", "Output")]
    for name, role in spec:
        v = raw_h1 if name == "h1_cm" else df[name].values
        if v is None:
            continue
        rows.append(dict(Variable=name, Role=role, Min=v.min(), Max=v.max(),
                         Mean=v.mean(), Sigma=v.std(),
                         CV_pct=100 * v.std() / v.mean(), Skew=stats.skew(v)))
    return pd.DataFrame(rows).round(3)


def zstats(a: np.ndarray):
    return a.mean(0, keepdims=True), a.std(0, keepdims=True) + 1e-8


# -----------------------------------------------------------------------------
# 10. C-PINN-ATT MODULES / CLASSES
# 11. ATTENTION
# 12. M1 / M2 / M3
# Source: cpinn_att_reproduce.py
# -----------------------------------------------------------------------------
# =============================================================================
# 2.  MODEL
# =============================================================================
class MHAttnBlock(nn.Module):
    """Multi-head self-attention over the projected token, residual + LayerNorm."""

    def __init__(self, d: int = 64, H: int = 4):
        super().__init__()
        self.H, self.dk = H, d // H
        self.q, self.k, self.v = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.o = nn.Linear(d, d)
        self.ln = nn.LayerNorm(d)

    def forward(self, x):
        B, d = x.shape
        q = self.q(x).view(B, self.H, self.dk)
        k = self.k(x).view(B, self.H, self.dk)
        v = self.v(x).view(B, self.H, self.dk)
        att = torch.softmax((q * k).sum(-1, keepdim=True) / self.dk ** 0.5, dim=1)
        ctx = (att * v).reshape(B, d)
        return self.ln(x + self.o(ctx)), att.mean(0).squeeze(-1)


class SubNet(nn.Module):
    """One PINN-Att sub-model (five-stage pipeline, Figure 8)."""

    def __init__(self, in_dim: int, H: int = 4, d: int = 64, dropout: float = 0.10,
                 attention: bool = True):
        super().__init__()
        self.attention = attention
        self.proj = nn.Linear(in_dim, d)
        self.attn = MHAttnBlock(d, H)
        self.ff = nn.Sequential(nn.Linear(d, 256), nn.Tanh(), nn.Linear(256, d))
        self.ln = nn.LayerNorm(d)
        self.head = nn.Sequential(
            nn.Linear(d, 128), nn.BatchNorm1d(128), nn.Tanh(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.Tanh(),
            nn.Linear(64, 32), nn.Tanh(),
            nn.Linear(32, 1))
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_normal_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, x):
        h = torch.tanh(self.proj(x))
        if self.attention:
            h, w = self.attn(h)
        else:
            w = torch.zeros(4, device=x.device)
        h = self.ln(h + self.ff(h))
        return self.head(h), w


class CPINNAtt(nn.Module):
    """Cascaded architecture M1/M2 -> M3 with hard routing (Figure 9)."""

    def __init__(self, H: int = 4, cascade: bool = True, attention: bool = True):
        super().__init__()
        self.cascade = cascade
        self.M1 = SubNet(3, H, dropout=0.08, attention=attention)
        self.M2 = SubNet(3, H, dropout=0.10, attention=attention)
        self.M3 = SubNet(5 if cascade else 3, H, dropout=0.12, attention=attention)

    def forward(self, x):
        Y, w1 = self.M1(x)
        S, w2 = self.M2(x)
        if self.cascade:
            # HARD ROUTING - Y and S are NOT detached: gradients flow back to M1/M2
            X, w3 = self.M3(torch.cat([x, Y, S], -1))
        else:
            X, w3 = self.M3(x)
        return Y, S, X, (w1, w2, w3)


# -----------------------------------------------------------------------------
# 7. HYDRAULIC RELATIONS
# 8. CONSTRAINTS C1, C2, C3, C4, C5, C6, C7
# 9. PVI  (momentum feasibility: measured Fr1, predicted Y)
# 15. METRICS
# Source: cpinn_att_reproduce.py
# -----------------------------------------------------------------------------
# =============================================================================
# 3.  PHYSICS CONSTRAINTS  (Table 3 of the manuscript)
# =============================================================================
def belanger(Fr1):
    """Eq. (3): free-jump sequent-depth ratio."""
    return 0.5 * (torch.sqrt(1 + 8 * Fr1 ** 2) - 1)


def w_gamma(g, gth: float = GAMMA_TH):
    """Regime-dependent activation: vanishes for gamma >= gth."""
    return torch.clamp(1 - g / gth, min=0.0) ** 2


def C1(Fr1, g, Y):
    """C1 [M1] gamma-weighted Belanger residual. Inactive for all compact data."""
    return w_gamma(g) * (Y - belanger(Fr1))


def C2(Fr1, Y):
    """C2 [M1] normalized momentum residual (Chaudhry, Eq. 6)."""
    r = 0.5 * (1 - Y ** 2) + Fr1 ** 2 * (1 - 1 / (Y + 1e-6))
    return r / (2 * Fr1 ** 2 + 1e-6)


def C6_feasibility(Fr1, Y):
    """C6 [cascade] asymmetric feasibility: penalize Fs < 0 only.

    NOTE (gradient-path verification, Section 4): Fs depends on Y_pred, i.e. on
    the output of M1, NOT on X_pred. Hence dL_C6/dtheta_M3 == 0 identically.
    This is the silent failure that motivates C7.
    """
    Fs = 0.5 * (1 - Y ** 2) + Fr1 ** 2 * (1 - 1 / (Y + 1e-6))
    return torch.clamp(-Fs, min=0.0)


def C7_forster_skrinde(Fr1, S, X):
    """C7 [M3] soft Forster-Skrinde coupling -- the direct physics gradient on M3.

    Used as a SOFT regularizer (not a hard equality) because the classical
    correlation does not hold quantitatively in the compact regime
    (R^2 = -0.255 against the measurements, Figure 5c).
    """
    return X - 5.8 * Fr1 ** 0.75 * torch.clamp(S, min=1e-4) ** 0.5


def C3_jump_length(Fr1, g, Y, k_lj: float = 6.5):
    """C3 [M1] Silvester classical jump-length scaling, in h1 units.

    The classical Silvester relation reads L_j/h1 = 9.75 (Fr1 - 1)^1.01.  Written
    that way it depends on Fr1 ALONE and therefore carries no gradient to any
    sub-model (the same failure mode that motivates C7).  We therefore close it
    on the predicted conjugate depth using the classical stilling-basin scaling
    L_j ~ 6-7 h2 (see Introduction and [1,17]), i.e. L_j/h1 = k_lj * Y, giving

        r_C3 = 9.75 (Fr1 - 1)^1.01  -  k_lj * Y_pred ,

    which does depend on Y_pred and hence delivers dL/dtheta_M1 != 0.
    Normalized by the classical length scale to keep the residual O(1).
    """
    Lj = 9.75 * torch.clamp(Fr1 - 1.0, min=1e-3) ** 1.01
    return (Lj - k_lj * Y) / (Lj + 1e-6)


def C4_hager_li(g, S, Fr1):
    """C4 [M2] Hager-Li sill-stability relation in the validated empirical form.

    Figure 5(b) shows that zeta(S,gamma) = (S + 0.48)/(1.07 - 0.29 gamma)
    collapses onto (Fr1 - 1) with R^2 = 0.854 on the present database.  The
    residual depends on S_pred, hence dL/dtheta_M2 != 0.
    """
    zeta = (S + 0.48) / (1.07 - 0.29 * g + 1e-6)
    return (zeta - (Fr1 - 1.0)) / (Fr1 + 1e-6)


def C5_admissibility(Y, S):
    """C5 [M2, cascade] geometric admissibility of the sill: 0 < s < h2.

    In h1 units this is 0 < S_pred < Y_pred; both bounds are satisfied by 100%
    of the measurements.  Implemented as a one-sided (hinge) penalty so that it
    is exactly zero on admissible predictions and grows quadratically outside.
    The upper bound depends on Y_pred as well, so the term also propagates a
    gradient through the cascade to M1.
    """
    lower = torch.clamp(-S, min=0.0)          # penalize S <= 0
    upper = torch.clamp(S - Y, min=0.0)       # penalize S >= Y
    return lower + upper


def feasibility_index(Fr1_np, Y_np) -> float:
    """PVI: percentage of predictions violating the momentum feasibility Fs >= 0."""
    Fs = 0.5 * (1 - Y_np ** 2) + Fr1_np ** 2 * (1 - 1 / (Y_np + 1e-6))
    return 100.0 * float(np.mean(Fs < 0))


# ---- additional goodness-of-fit metrics (added, not replacing existing ones) --
def mape(obs: np.ndarray, pred: np.ndarray) -> float:
    """Mean Absolute Percentage Error (%). Guards against division by zero."""
    obs = np.asarray(obs, dtype=float)
    denom = np.where(np.abs(obs) < 1e-12, 1e-12, obs)
    return 100.0 * float(np.mean(np.abs((obs - pred) / denom)))


def nse(obs: np.ndarray, pred: np.ndarray) -> float:
    """Nash-Sutcliffe Efficiency. NSE = 1 - SS_res / SS_tot (=R2 for regression)."""
    obs = np.asarray(obs, dtype=float)
    ss_res = float(np.sum((obs - pred) ** 2))
    ss_tot = float(np.sum((obs - obs.mean()) ** 2)) + 1e-12
    return 1.0 - ss_res / ss_tot


# Loss weights that produced the numbers reported in the manuscript.
# C3 (Silvester), C4 (Hager-Li) and C5 (continuity) require jump-length /
# Lambda measurements that are not tabulated in the present database; they are
# therefore declared but carry zero weight in the reported configuration.
# All nine constraints are active.  C3/C4/C5 were reformulated so that each
# one depends on a PREDICTED quantity and therefore carries a real gradient
# (gradient-path verification, Section 4 of the manuscript).
LAMBDAS_PAPER = dict(data=1.0, c1=0.1, c2=0.1, c3=0.05, c4=0.05,
                     c5=0.5, c6=0.5, c7=0.05)


# -----------------------------------------------------------------------------
# 13. LOSS
# 14. TRAINING / PREDICTION
# Source: cpinn_att_reproduce.py
# -----------------------------------------------------------------------------
# =============================================================================
# 4.  TRAINING
# =============================================================================
def run_cpinn(df, tri, tei, n_epochs=2000, use_gamma=True, cascade=True,
              attention=True, lambdas=None, return_model=False, seed=SEED):
    """Train C-PINN-Att on rows `tri`, evaluate on rows `tei`.

    Returns dict with R2/RMSE/MAE per output, PVI, predictions and (optionally)
    the trained model plus the normalization constants.
    """
    torch.manual_seed(seed); np.random.seed(seed)
    lam = dict(LAMBDAS_PAPER if lambdas is None else lambdas)

    tr, te = df.iloc[tri], df.iloc[tei]
    Xtr = tr[FEATS].values.astype(np.float32).copy()
    Xte = te[FEATS].values.astype(np.float32).copy()
    if not use_gamma:                       # gamma-free ablation (Section 3.2)
        Xtr[:, 2] = 0.0; Xte[:, 2] = 0.0
    Ytr = tr[OUTS].values.astype(np.float32)
    Yte = te[OUTS].values.astype(np.float32)

    mu_x, sd_x = zstats(Xtr); mu_y, sd_y = zstats(Ytr)
    Xtr_n = torch.tensor((Xtr - mu_x) / sd_x)
    Xte_n = torch.tensor((Xte - mu_x) / sd_x)
    Ytr_n = torch.tensor((Ytr - mu_y) / sd_y)
    muy, sdy = torch.tensor(mu_y), torch.tensor(sd_y)
    Fr_tr = torch.tensor(Xtr[:, 0:1]); g_tr = torch.tensor(Xtr[:, 2:3])
    varX = float(df["X"].var())

    model = CPINNAtt(cascade=cascade, attention=attention)
    opt = torch.optim.Adam(model.parameters(), lr=8e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)

    # loss history (recording only; does not affect the optimization)
    hist = {"epoch": [], "total": [], "data": [], "physics": []}

    for ep_i in range(n_epochs):
        model.train()
        Yh, Sh, Xh, _ = model(Xtr_n)
        L_data = F.mse_loss(torch.cat([Yh, Sh, Xh], -1), Ytr_n)
        # de-normalize for physics evaluated in physical units
        Yp = Yh * sdy[0, 0] + muy[0, 0]
        Sp = Sh * sdy[0, 1] + muy[0, 1]
        Xp = Xh * sdy[0, 2] + muy[0, 2]
        L_eq = (lam["c1"] * (C1(Fr_tr, g_tr, Yp) ** 2).mean()
                + lam["c2"] * (C2(Fr_tr, Yp) ** 2).mean()
                + lam["c3"] * (C3_jump_length(Fr_tr, g_tr, Yp) ** 2).mean()
                + lam["c4"] * (C4_hager_li(g_tr, Sp, Fr_tr) ** 2).mean()
                + lam["c5"] * (C5_admissibility(Yp, Sp) ** 2).mean()
                + lam["c6"] * (C6_feasibility(Fr_tr, Yp) ** 2).mean()
                + lam["c7"] * (C7_forster_skrinde(Fr_tr, Sp, Xp) ** 2).mean() / varX)
        loss = lam["data"] * L_data + L_eq
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step(); sch.step()
        hist["epoch"].append(ep_i + 1)
        hist["total"].append(float(loss.item()))
        hist["data"].append(float((lam["data"] * L_data).item()))
        hist["physics"].append(float(L_eq.item()))

    model.eval()
    with torch.no_grad():
        Yh, Sh, Xh, _ = model(Xte_n)
        pred = (torch.cat([Yh, Sh, Xh], -1) * sdy + muy).numpy()

    out = dict(
        R2=[float(r2_score(Yte[:, i], pred[:, i])) for i in range(3)],
        RMSE=[float(np.sqrt(mean_squared_error(Yte[:, i], pred[:, i]))) for i in range(3)],
        MAE=[float(mean_absolute_error(Yte[:, i], pred[:, i])) for i in range(3)],
        MAPE=[mape(Yte[:, i], pred[:, i]) for i in range(3)],
        NSE=[nse(Yte[:, i], pred[:, i]) for i in range(3)],
        PVI=feasibility_index(Xte[:, 0], pred[:, 0]),
        pred=pred, obs=Yte, test_rows=te, loss_history=hist)
    if return_model:
        out.update(model=model, norm=(mu_x, sd_x, mu_y, sd_y), Xte_n=Xte_n)
    return out


def run_baseline(df, tri, tei, kind="dnn"):
    """Deep-network baseline used in Tables 4 and 6, under an identical protocol.

    Only the "dnn" baseline is retained here. The obsolete ELM / SVR / RBF-NN
    branches belonged to the superseded Table 7 routine and are not part of the
    final manuscript; the final benchmark is run_benchmark() further below.
    One regressor is fitted independently per output, as in the source script.
    """
    if kind != "dnn":
        raise ValueError(
            f"Unsupported baseline {kind!r}. Only 'dnn' is retained in the final "
            "implementation; the final benchmark is provided by run_benchmark().")
    tr, te = df.iloc[tri], df.iloc[tei]
    Xtr, Xte = tr[FEATS].values, te[FEATS].values
    Ytr, Yte = tr[OUTS].values, te[OUTS].values
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-8
    Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd
    pred = np.zeros_like(Yte, dtype=float)
    for i in range(3):
        m = MLPRegressor(hidden_layer_sizes=(256, 128, 64, 32), activation="tanh",
                         max_iter=3000, random_state=SEED, alpha=1e-3)
        m.fit(Xtr, Ytr[:, i]); pred[:, i] = m.predict(Xte)
    return dict(
        R2=[float(r2_score(Yte[:, i], pred[:, i])) for i in range(3)],
        RMSE=[float(np.sqrt(mean_squared_error(Yte[:, i], pred[:, i]))) for i in range(3)],
        PVI=feasibility_index(te[FEATS].values[:, 0], pred[:, 0]),
        pred=pred, obs=Yte)


def folds_of(df, k=5):
    return list(StratifiedKFold(k, shuffle=True, random_state=SEED)
                .split(df.index, df["gamma_class"].values))


# -----------------------------------------------------------------------------
# 16. FIXED 114/37 EVALUATION
# 17. CROSS-VALIDATION ROUTINES
# 18. ABLATION ROUTINES
# Source: cpinn_att_reproduce.py
# NOTE: the obsolete exp_table7() routine (ELM/DNN/SVR/RBF-NN) is deliberately
#       NOT included. The final Table 7 benchmark is implemented below from
#       benchmark_models.py, which is authoritative.
# -----------------------------------------------------------------------------
# =============================================================================
# 5.  EXPERIMENTS
# =============================================================================
def exp_table4(df, folds, ep, R):
    """Table 4 + Sections 3.2 / 3.3 paired t-tests."""
    print("\n=== Table 4 : stratified 5-fold CV ===")
    acc = {k: {o: [] for o in OUTS + ["PVI"]} for k in ["DNN", "gamma_free", "CPINN"]}
    for tri, tei in folds:
        for key, fn in [("DNN", lambda: run_baseline(df, tri.tolist(), tei.tolist(), "dnn")),
                        ("gamma_free", lambda: run_cpinn(df, tri.tolist(), tei.tolist(), ep, use_gamma=False)),
                        ("CPINN", lambda: run_cpinn(df, tri.tolist(), tei.tolist(), ep))]:
            r = fn()
            for i, o in enumerate(OUTS):
                acc[key][o].append(r["R2"][i])
            acc[key]["PVI"].append(r["PVI"])
    for key in ["DNN", "gamma_free", "CPINN"]:
        a = acc[key]
        print(f"  {key:11s} " + "  ".join(
            f"R2({o})={np.mean(a[o]):.3f}±{np.std(a[o]):.3f}" for o in OUTS)
            + f"  PVI={np.mean(a['PVI']):.1f}±{np.std(a['PVI']):.1f}")
    R["table4"] = {k: {o: [float(np.mean(v[o])), float(np.std(v[o]))]
                       for o in OUTS + ["PVI"]} for k, v in acc.items()}
    R["table4_raw"] = {k: {o: [float(x) for x in v[o]] for o in OUTS + ["PVI"]}
                       for k, v in acc.items()}

    print("\n=== Section 3.2 : gamma-conditioning ablation (paired t-test) ===")
    tt = {}
    for o in OUTS:
        t, p = stats.ttest_rel(acc["CPINN"][o], acc["gamma_free"][o])
        sig = "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "NS"
        tt[o] = [float(t), float(p), sig]
        print(f"  R2({o}): full={np.mean(acc['CPINN'][o]):.3f} "
              f"free={np.mean(acc['gamma_free'][o]):.3f}  t={t:.2f} p={p:.4f} {sig}")
    R["ttest_gamma_free"] = tt

    print("\n=== Section 3.3 : C-PINN-Att vs DNN baseline (paired t-test) ===")
    tt2 = {}
    for o in OUTS:
        t, p = stats.ttest_rel(acc["CPINN"][o], acc["DNN"][o])
        sig = "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "NS"
        tt2[o] = [float(t), float(p), sig]
        print(f"  R2({o}): t={t:.2f} p={p:.4f} {sig}")
    R["ttest_dnn"] = tt2
    return acc


def exp_table5(df, folds, ep, R):
    """Table 5 : per-gamma-class cross-validated performance."""
    print("\n=== Table 5 : per-gamma-class CV ===")
    per = {c: {o: [] for o in OUTS} | {"n": []} for c in GORDER}
    for tri, tei in folds:
        r = run_cpinn(df, tri.tolist(), tei.tolist(), ep)
        te = r["test_rows"].reset_index(drop=True)
        for c in GORDER:
            m = (te["gamma_class"] == c).values
            if m.sum() >= 2:
                for i, o in enumerate(OUTS):
                    per[c][o].append(r2_score(r["obs"][m, i], r["pred"][m, i]))
                per[c]["n"].append(int(m.sum()))
    tab = {}
    for c in GORDER:
        d = per[c]
        tab[c] = dict(n=float(np.mean(d["n"])) if d["n"] else 0.0,
                      **{o: (float(np.mean(d[o])) if d[o] else None) for o in OUTS})
        vals = "  ".join(f"R2({o})={tab[c][o]:.3f}" if tab[c][o] is not None else f"R2({o})=n/a"
                         for o in OUTS)
        print(f"  {c}: n={tab[c]['n']:.1f}  {vals}")
    R["table5"] = tab


def exp_table6(df, tr, te, ep, R):
    """Table 6 : progressive architectural ablation on the fixed split."""
    print("\n=== Table 6 : progressive ablation (fixed 114/37 split) ===")
    L = dict(LAMBDAS_PAPER)
    stages = [
        ("DNN baseline",          lambda: run_baseline(df, tr, te, "dnn")),
        ("PINN-A (+C1,C2)",       lambda: run_cpinn(df, tr, te, ep, attention=False, cascade=False,
                                                    lambdas={**L, "c6": 0.0, "c7": 0.0})),
        ("PINN-B (+C6)",          lambda: run_cpinn(df, tr, te, ep, attention=False, cascade=False,
                                                    lambdas={**L, "c7": 0.0})),
        ("PINN-C (+C7)",          lambda: run_cpinn(df, tr, te, ep, attention=False, cascade=False)),
        ("PINN-Att (+Attention)", lambda: run_cpinn(df, tr, te, ep, attention=True, cascade=False)),
        ("C-PINN-Att (+Cascade)", lambda: run_cpinn(df, tr, te, ep, attention=True, cascade=True)),
    ]
    out = {}
    for name, fn in stages:
        r = fn()
        out[name] = [float(np.mean(r["R2"])), float(np.mean(r["RMSE"])), float(r["PVI"])]
        print(f"  {name:24s} R2={out[name][0]:.3f}  RMSE={out[name][1]:.2f}  PVI={out[name][2]:.1f}%")
    R["table6"] = out

# =============================================================================
# 6.  SECTION 3.8  --  VALIDATION STRATEGY
# =============================================================================
def exp_multiseed(df, tr, te, ep, R, seeds=(42, 0, 1, 7, 123)):
    """Multi-seed robustness of the fixed-split results and of the diagnostic.

    Two distinct questions are answered separately.

    (a) Performance.  The reported metrics come from a single initialisation
        (SEED = 42).  Re-running the same protocol with different seeds
        quantifies how much of the reported value is initialisation-dependent.

    (b) The gradient-path diagnostic.  The zero entry of C6 on theta_M3 is a
        property of the computational graph, not of the initialisation, so it
        must hold for EVERY seed.  This is checked explicitly: any non-zero
        value would invalidate the central claim of the paper.
    """
    print("\n=== Multi-seed robustness (fixed 114/37 split) ===")
    acc = {"Y": [], "S": [], "X": [], "PVI": []}
    for sd in seeds:
        r = run_cpinn(df, tr, te, n_epochs=ep, seed=sd)
        for i, o in enumerate(OUTS):
            acc[o].append(r["R2"][i])
        acc["PVI"].append(r["PVI"])
        print(f"  seed {sd:>4d}: " + "  ".join(f"R2({o})={r['R2'][i]:.3f}"
              for i, o in enumerate(OUTS)) + f"  PVI={r['PVI']:.1f}%", flush=True)
    summ = {o: [float(np.mean(acc[o])), float(np.std(acc[o]))] for o in OUTS + ["PVI"]}
    print("  ---")
    print("  mean +- std : " + "  ".join(
        f"R2({o})={summ[o][0]:.3f}+-{summ[o][1]:.3f}" for o in OUTS)
        + f"  PVI={summ['PVI'][0]:.1f}+-{summ['PVI'][1]:.1f}%")
    R["multiseed_performance"] = {"seeds": list(seeds), "raw": acc, "summary": summ}

    # ---- (b) the diagnostic must be seed-independent ------------------------
    print("\n=== Seed-independence of the gradient-path diagnostic ===")
    print(f"  {'seed':>5s} {'||dC6/dM1||':>13s} {'||dC6/dM3||':>13s} {'||dC7/dM3||':>13s}")
    diag = []
    Xall = torch.tensor(df[FEATS].values.astype(np.float32))
    Xn = (Xall - Xall.mean(0, keepdim=True)) / (Xall.std(0, keepdim=True) + 1e-8)
    Fr = Xall[:, 0:1]
    for sd in seeds:
        torch.manual_seed(sd); np.random.seed(sd)
        mod = CPINNAtt()
        def gnorm(term, params):
            mod.zero_grad()
            Y, S, Xh, _ = mod(Xn)
            gr = torch.autograd.grad(term(Y, S, Xh), params, allow_unused=True)
            return float(np.sqrt(sum((x ** 2).sum().item() for x in gr if x is not None)))
        c6 = lambda Y, S, Xh: (C6_feasibility(Fr, Y) ** 2).mean()
        c7 = lambda Y, S, Xh: (C7_forster_skrinde(Fr, S, Xh) ** 2).mean()
        g61 = gnorm(c6, list(mod.M1.parameters()))
        g63 = gnorm(c6, list(mod.M3.parameters()))
        g73 = gnorm(c7, list(mod.M3.parameters()))
        diag.append({"seed": sd, "C6_M1": g61, "C6_M3": g63, "C7_M3": g73})
        print(f"  {sd:>5d} {g61:13.3e} {g63:13.3e} {g73:13.3e}")
    ok = all(d["C6_M3"] == 0.0 for d in diag) and all(d["C7_M3"] > 0 for d in diag)
    print(f"  -> C6 delivers exactly zero to M3 for every seed: {ok}")
    print("     (the diagnostic is a property of the graph, not of the initialisation)")
    R["multiseed_diagnostic"] = {"rows": diag, "claim_holds": bool(ok)}


def exp_section38(df, tr, te, ep, R, outdir):
    """Table 8 + residual diagnostics + MC-dropout calibration + C-1..C-4."""
    print("\n=== Section 3.8 : validation strategy (fixed split) ===")
    r = run_cpinn(df, tr, te, ep, return_model=True)
    pred, obs = r["pred"], r["obs"]

    # ---- 3.8.1.1 point metrics + non-parametric bootstrap CIs --------------
    rng = np.random.default_rng(SEED)
    tab8 = {}
    for i, o in enumerate(OUTS):
        boots = [r2_score(obs[idx, i], pred[idx, i]) for idx in
                 (rng.integers(0, len(obs), len(obs)) for _ in range(2000))]
        lo, hi = np.percentile(boots, [2.5, 97.5])
        rng_span = obs[:, i].max() - obs[:, i].min()
        tab8[o] = dict(R2=r["R2"][i], CI=[float(lo), float(hi)], RMSE=r["RMSE"][i],
                       NRMSE_pct=100 * r["RMSE"][i] / rng_span, MAE=r["MAE"][i])
        print(f"  {o}: R2={r['R2'][i]:.3f} 95%CI=[{lo:.3f},{hi:.3f}] "
              f"RMSE={r['RMSE'][i]:.3f} NRMSE={tab8[o]['NRMSE_pct']:.2f}% MAE={r['MAE'][i]:.3f}")
    R["table8"] = tab8
    R["fixed_PVI"] = r["PVI"]
    print(f"  PVI (fixed split) = {r['PVI']:.1f}%")

    # ---- 3.8.1.2 residual normality ---------------------------------------
    resid = obs - pred
    sw = {}
    for i, o in enumerate(OUTS):
        W, p = stats.shapiro(resid[:, i])
        sw[o] = [float(W), float(p)]
    print("  Shapiro-Wilk: " + ",  ".join(
        f"{o}: W={sw[o][0]:.2f} (p={sw[o][1]:.1e})" for o in OUTS))
    R["shapiro"] = sw

    # ---- 3.8.1.2 MC-dropout calibration -----------------------------------
    model, Xte_n = r["model"], r["Xte_n"]
    _, _, mu_y, sd_y = r["norm"]
    model.eval()
    for m in model.modules():                 # enable dropout only
        if isinstance(m, nn.Dropout):
            m.train()
    T = 200
    samples = np.zeros((T, len(te), 3))
    with torch.no_grad():
        for t in range(T):
            Yh, Sh, Xh, _ = model(Xte_n)
            samples[t] = (torch.cat([Yh, Sh, Xh], -1).numpy() * sd_y + mu_y)
    mc_mean, mc_std = samples.mean(0), samples.std(0) + 1e-9
    cov = {}
    for i, o in enumerate(OUTS):
        z = np.abs(obs[:, i] - mc_mean[:, i]) / mc_std[:, i]
        cov[o] = [float(100 * np.mean(z <= 1)), float(100 * np.mean(z <= 2))]
        print(f"  MC-dropout coverage {o}: ±1σ={cov[o][0]:.1f}%  ±2σ={cov[o][1]:.1f}%"
              f"   (nominal 68.3 / 95.4)")
    R["calibration"] = cov

    # ---- 3.8.2 functional consistency with canonical correlations ----------
    print("  --- functional consistency (full dataset) ---")
    Fr = df["Fr1"].values; g = df["gamma"].values
    Yv, Sv, Xv = df["Y"].values, df["S"].values, df["X"].values
    # predictions on the whole dataset for the checks
    rall = run_cpinn(df, list(range(len(df))), list(range(len(df))), ep)
    pall = rall["pred"]

    Yb = 0.5 * (np.sqrt(1 + 8 * Fr ** 2) - 1)          # C-1 Belanger
    dev = (pall[:, 0] - Yb) / Yb
    rho, prho = stats.spearmanr(g, dev)
    fs = 5.8 * Fr ** 0.75 * Sv ** 0.5                  # C-4 Forster-Skrinde
    func = dict(belanger_mean_dev_pct=float(dev.mean() * 100),
                belanger_spearman_rho=float(rho), belanger_spearman_p=float(prho),
                fs_R2_observed=float(r2_score(Xv, fs)),
                fs_R2_predicted=float(r2_score(pall[:, 2], fs)))
    print(f"  C-1 Belanger : mean dev = {func['belanger_mean_dev_pct']:.1f}%, "
          f"Spearman rho = {rho:.2f} (p={prho:.1e})")
    print(f"  C-2 Hager-Bremen-Kawagoshi : satisfied by construction (gamma = Lr*/x)")
    print(f"  C-3 Hager-Li : dataset outside original calibration range (b=0.295 m)")
    print(f"  C-4 Forster-Skrinde : R2 vs observed = {func['fs_R2_observed']:.3f}, "
          f"vs predicted = {func['fs_R2_predicted']:.3f}")
    R["functional_consistency"] = func

    # ---- extended global metrics on the fixed split (MAPE, NSE added) ------
    R["fixed_metrics"] = {o: dict(
        R2=r["R2"][i], RMSE=r["RMSE"][i], MAE=r["MAE"][i],
        MAPE=r["MAPE"][i], NSE=r["NSE"][i]) for i, o in enumerate(OUTS)}

    # ---- per-sample observed / predicted / abs & rel error table ----------
    per_rows = []
    tr_rows = r["test_rows"].reset_index(drop=True)
    for j in range(len(obs)):
        row = {"sample": int(j + 1),
               "gamma_class": tr_rows["gamma_class"].iloc[j],
               "Fr1": float(tr_rows["Fr1"].iloc[j])}
        for i, o in enumerate(OUTS):
            ov, pv = float(obs[j, i]), float(pred[j, i])
            row[f"{o}_obs"] = round(ov, 4)
            row[f"{o}_pred"] = round(pv, 4)
            row[f"{o}_abs_err"] = round(abs(ov - pv), 4)
            row[f"{o}_rel_err_%"] = round(100 * abs(ov - pv) / (abs(ov) + 1e-12), 2)
        per_rows.append(row)
    R["obs_pred_table"] = per_rows

    # ---- persist loss history for the training-curve figure ---------------
    R["loss_history"] = r["loss_history"]

    np.savez(os.path.join(outdir, "fixed_split_predictions.npz"),
             pred=pred, obs=obs, mc_std=mc_std,
             gclass=r["test_rows"]["gamma_class"].values,
             Fr1=r["test_rows"]["Fr1"].values)

    # ---- save the trained model (weights + full state) --------------------
    models_dir = os.path.join(outdir, "models")
    os.makedirs(models_dir, exist_ok=True)
    try:
        torch.save(r["model"].state_dict(),
                   os.path.join(models_dir, "cpinn_att_fixed_split.pt"))
    except Exception as exc:
        print(f"  [warn] could not save model state_dict: {exc}")
    return r


# -----------------------------------------------------------------------------
# 19. FINAL BENCHMARK IMPLEMENTATIONS
# 20. FIVE-SEED BENCHMARK  (seeds 42, 0, 1, 7, 123)
# Source: benchmark_models.py  (AUTHORITATIVE for the final Table 7)
# Baselines are fitted with one separate regressor per output; the three
# predictions are assembled afterwards. They are not joint multi-output models.
# -----------------------------------------------------------------------------
# ----------------------------------------------------------------- ANFIS
class ANFIS(nn.Module):
    """First-order Takagi-Sugeno ANFIS with Gaussian membership functions.

    n_mf membership functions per input give n_mf**n_in rules. Two functions
    on three inputs yield eight rules, which is the usual choice at N = 151;
    larger rule bases overfit.
    """

    def __init__(self, n_in=3, n_mf=2):
        super().__init__()
        self.n_in, self.n_rules = n_in, n_mf ** n_in
        self.c = nn.Parameter(torch.linspace(-1.5, 1.5, n_mf).repeat(n_in, 1).clone())
        self.s = nn.Parameter(torch.ones(n_in, n_mf))
        self.p = nn.Parameter(torch.zeros(self.n_rules, n_in))
        self.r = nn.Parameter(torch.zeros(self.n_rules))
        idx = np.array(np.meshgrid(*[range(n_mf)] * n_in, indexing="ij")).reshape(n_in, -1).T
        self.register_buffer("idx", torch.tensor(idx, dtype=torch.long))

    def forward(self, x):
        mu = torch.exp(-((x.unsqueeze(-1) - self.c.unsqueeze(0)) ** 2)
                       / (2 * self.s.unsqueeze(0) ** 2 + 1e-8))
        g = torch.stack([mu[:, i, self.idx[:, i]] for i in range(self.n_in)], dim=1)
        w = g.prod(dim=1)
        wn = w / (w.sum(dim=1, keepdim=True) + 1e-9)
        return (wn * (x @ self.p.T + self.r)).sum(dim=1, keepdim=True)


def fit_anfis(Xtr, ytr, Xte, seed, n_epochs=3000):
    torch.manual_seed(seed); np.random.seed(seed)
    xt = torch.tensor(Xtr, dtype=torch.float32)
    yt = torch.tensor(ytr, dtype=torch.float32).reshape(-1, 1)
    m = ANFIS(n_in=Xtr.shape[1])
    opt = torch.optim.Adam(m.parameters(), lr=2e-2)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    for _ in range(n_epochs):
        loss = ((m(xt) - yt) ** 2).mean()
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 5.0)
        opt.step(); sch.step()
    m.eval()
    with torch.no_grad():
        return m(torch.tensor(Xte, dtype=torch.float32)).numpy().ravel()


# ----------------------------------------------------------------- baselines
def fit_xgb(Xtr, ytr, Xte, seed):
    from xgboost import XGBRegressor
    m = XGBRegressor(n_estimators=400, max_depth=3, learning_rate=0.05,
                     subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
                     random_state=seed, verbosity=0)
    m.fit(Xtr, ytr)
    return m.predict(Xte)


def fit_rf(Xtr, ytr, Xte, seed):
    m = RandomForestRegressor(n_estimators=500, max_depth=None,
                              min_samples_leaf=2, random_state=seed, n_jobs=-1)
    m.fit(Xtr, ytr)
    return m.predict(Xte)


def fit_mlp(Xtr, ytr, Xte, seed):
    m = MLPRegressor(hidden_layer_sizes=(128, 64, 32), activation="tanh",
                     solver="adam", learning_rate_init=8e-4, max_iter=4000,
                     alpha=1e-3, random_state=seed, early_stopping=False)
    m.fit(Xtr, ytr)
    return m.predict(Xte)


MODELS = {"ANFIS": (fit_anfis, True), "XGBoost": (fit_xgb, False),
          "Random Forest": (fit_rf, False), "MLP (no physics)": (fit_mlp, True)}


def pvi(Fr, Y):
    """Physical-violation index: share of predictions with negative specific force."""
    Fs = 0.5 * (1 - Y ** 2) + Fr ** 2 * (1 - 1 / (Y + 1e-6))
    return 100.0 * float(np.mean(Fs < 0))


# NOTE: renamed from run() to run_benchmark() to avoid a name collision
# inside a single consolidated cell. Logic is unchanged.
def run_benchmark(xlsx=DATA_FILE, seeds=SEEDS):
    df = pd.ExcelFile(xlsx).parse("Adimensional_A")
    tr = df.index[df["Split"] == "Train"].to_numpy()
    te = df.index[df["Split"] == "Test"].to_numpy()
    Xtr_raw, Xte_raw = df.loc[tr, FEATS].values, df.loc[te, FEATS].values
    mu, sd = Xtr_raw.mean(0), Xtr_raw.std(0) + 1e-8
    Xtr_n, Xte_n = (Xtr_raw - mu) / sd, (Xte_raw - mu) / sd
    Ytr, Yte = df.loc[tr, OUTS].values, df.loc[te, OUTS].values

    print("=" * 78)
    print(f" MULTI-MODEL BENCHMARK — fixed {len(tr)}/{len(te)} split, seeds {list(seeds)}")
    print("=" * 78)

    acc = {n: {"R2": [], "RMSE": [], "MAE": [], "PVI": [], "T": []} for n in MODELS}
    for seed in seeds:
        for name, (fn, needs_norm) in MODELS.items():
            Xa, Xb = (Xtr_n, Xte_n) if needs_norm else (Xtr_raw, Xte_raw)
            t0 = time.perf_counter()
            pred = np.column_stack([fn(Xa, Ytr[:, i], Xb, seed) for i in range(3)])
            dt = time.perf_counter() - t0
            acc[name]["R2"].append([r2_score(Yte[:, i], pred[:, i]) for i in range(3)])
            acc[name]["RMSE"].append([np.sqrt(mean_squared_error(Yte[:, i], pred[:, i]))
                                      for i in range(3)])
            acc[name]["MAE"].append([mean_absolute_error(Yte[:, i], pred[:, i])
                                     for i in range(3)])
            acc[name]["PVI"].append(pvi(Xte_raw[:, 0], pred[:, 0]))
            acc[name]["T"].append(dt)
        print(f"  seed {seed} done", flush=True)

    print(f"\n{'Model':17s} {'R2(Y)':>13s} {'R2(S)':>13s} {'R2(X)':>13s} "
          f"{'RMSE':>7s} {'MAE':>7s} {'PVI%':>6s} {'t(s)':>7s}")
    rows = []
    for name in MODELS:
        R2 = np.array(acc[name]["R2"]); RM = np.array(acc[name]["RMSE"])
        MA = np.array(acc[name]["MAE"]); PV = np.array(acc[name]["PVI"])
        T = np.array(acc[name]["T"])
        print(f"{name:17s} " + " ".join(f"{R2[:,i].mean():.3f}\u00b1{R2[:,i].std():.3f}"
              for i in range(3))
              + f" {RM.mean():7.3f} {MA.mean():7.3f} {PV.mean():6.1f} {T.mean():7.1f}")
        rows.append(dict(model=name,
                         R2=[float(R2[:, i].mean()) for i in range(3)],
                         R2_sd=[float(R2[:, i].std()) for i in range(3)],
                         RMSE=float(RM.mean()), MAE=float(MA.mean()),
                         PVI=float(PV.mean()), time_s=float(T.mean())))
    json.dump(rows, open("benchmark_full.json", "w"), indent=2)
    print("\nsaved benchmark_full.json")
    return rows


# -----------------------------------------------------------------------------
# 22. FIGURE GENERATION
# Source: cpinn_att_reproduce.py
# -----------------------------------------------------------------------------
# =============================================================================
# 7.  FIGURES
# =============================================================================
def make_figures(df, acc, R, outdir, ep):
    """Data figures (2-6) and result figures (12-17)."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from scipy.stats import gaussian_kde

    colors = dict(zip(GORDER, plt.cm.viridis(np.linspace(0, 0.9, 6))))
    labels = ["Y = h\u2082/h\u2081", "S = s/h\u2081", "X = x/h\u2081"]

    # ---- Figure 2 : distributions ----
    fig, axs = plt.subplots(2, 3, figsize=(12, 6))
    for ax, (v, lab) in zip(axs[0], [("Fr1", "Fr\u2081"), ("h1_n", "h\u2081/\u0125\u2081"), ("gamma", "\u03b3")]):
        d = df[v].values
        ax.hist(d, bins=18, density=True, color="#4C72B0", alpha=.6, edgecolor="k", lw=.3)
        xs = np.linspace(d.min(), d.max(), 200); ax.plot(xs, gaussian_kde(d)(xs), "k-", lw=1.5)
        ax.set_title(lab); ax.set_ylabel("density")
        if v == "gamma": ax.axvline(GAMMA_TH, ls="--", color="r", lw=1)
    for ax, (v, lab) in zip(axs[1], zip(OUTS, labels)):
        d = df[v].values
        ax.hist(d, bins=18, density=True, color="#C44E52", alpha=.6, edgecolor="k", lw=.3)
        xs = np.linspace(d.min(), d.max(), 200); ax.plot(xs, gaussian_kde(d)(xs), "k-", lw=1.5)
        ax.set_title(lab); ax.set_ylabel("density")
    fig.suptitle("Figure 2. Probability density distributions of inputs (top) and outputs (bottom)")
    fig.tight_layout(); fig.savefig(f"{outdir}/Figure02.png", dpi=300); plt.close()

    # ---- Figure 3 : correlation matrix ----
    cols = FEATS + OUTS; labs = ["Fr\u2081", "h\u2081", "\u03b3"] + OUTS
    C = df[cols].corr().values
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(C, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(6)); ax.set_yticks(range(6))
    ax.set_xticklabels(labs); ax.set_yticklabels(labs)
    for i in range(6):
        for j in range(6):
            ax.text(j, i, f"{C[i, j]:.2f}", ha="center", va="center", fontsize=9,
                    color="white" if abs(C[i, j]) > .5 else "black")
    ax.axhline(2.5, color="k", lw=2); ax.axvline(2.5, color="k", lw=2)
    fig.colorbar(im, ax=ax, shrink=.8); ax.set_title("Figure 3. Pearson correlation matrix")
    fig.tight_layout(); fig.savefig(f"{outdir}/Figure03.png", dpi=300); plt.close()

    # ---- Figure 5 : physical alignment ----
    fig, axs = plt.subplots(1, 3, figsize=(14, 4))
    psi = 1.386 * df["Fr1"] - 2 * np.log(df["gamma"]) - 0.617
    axs[0].scatter(psi, df["Y"], c=df["gamma"], cmap="viridis", s=25, edgecolor="k", lw=.3)
    lo, hi = min(psi.min(), df["Y"].min()), max(psi.max(), df["Y"].max())
    axs[0].plot([lo, hi], [lo, hi], "k-")
    axs[0].set_xlabel("\u03c8(Fr\u2081,\u03b3)"); axs[0].set_ylabel("Observed Y")
    axs[0].set_title(f"(a) Y vs \u03c8  R\u00b2={r2_score(df['Y'], psi):.3f}")
    zeta = (df["S"] + 0.48) / (1.07 - 0.29 * df["gamma"])
    axs[1].scatter(df["Fr1"] - 1, zeta, c=df["gamma"], cmap="viridis", s=25, edgecolor="k", lw=.3)
    lo, hi = min((df["Fr1"] - 1).min(), zeta.min()), max((df["Fr1"] - 1).max(), zeta.max())
    axs[1].plot([lo, hi], [lo, hi], "k-")
    axs[1].set_xlabel("Fr\u2081 \u2212 1"); axs[1].set_ylabel("\u03b6(S,\u03b3)")
    axs[1].set_title(f"(b) \u03b6 vs Fr\u2081\u22121  R\u00b2={r2_score(df['Fr1']-1, zeta):.3f}")
    fsv = 5.8 * df["Fr1"] ** 0.75 * df["S"] ** 0.5
    sc = axs[2].scatter(fsv, df["X"], c=df["gamma"], cmap="viridis", s=25, edgecolor="k", lw=.3)
    lo, hi = min(fsv.min(), df["X"].min()), max(fsv.max(), df["X"].max())
    axs[2].plot([lo, hi], [lo, hi], "k-")
    axs[2].set_xlabel("5.8\u00b7Fr\u2081$^{3/4}$\u00b7S$^{1/2}$"); axs[2].set_ylabel("Observed X")
    axs[2].set_title(f"(c) Forster\u2013Skrinde  R\u00b2={r2_score(df['X'], fsv):.3f}")
    fig.colorbar(sc, ax=axs[2], label="\u03b3")
    fig.suptitle("Figure 5. Physical alignment with established hydraulic correlations")
    fig.tight_layout(); fig.savefig(f"{outdir}/Figure05.png", dpi=300); plt.close()

    # ---- Figure 6 : gamma as governing variable ----
    fig, axs = plt.subplots(2, 2, figsize=(11, 8))
    gs = np.linspace(0, 2.3, 300)
    axs[0, 0].plot(gs, np.clip(1 - gs / GAMMA_TH, 0, None) ** 2, "b-", lw=2)
    axs[0, 0].axvline(GAMMA_TH, ls="--", color="r")
    axs[0, 0].set_xlabel("\u03b3"); axs[0, 0].set_ylabel("w(\u03b3)")
    axs[0, 0].set_title("(a) Activation w(\u03b3)")
    for c in GORDER:
        m = df["gamma_class"] == c
        axs[0, 1].scatter(df.loc[m, "gamma"], df.loc[m, "Y"], color=colors[c], s=22, label=c)
        axs[1, 0].scatter(df.loc[m, "gamma"], df.loc[m, "S"], color=colors[c], s=22)
    axs[0, 1].set_xlabel("\u03b3"); axs[0, 1].set_ylabel("Y"); axs[0, 1].legend(fontsize=6)
    axs[0, 1].set_title("(b) Y vs \u03b3 by class")
    axs[1, 0].set_xlabel("\u03b3"); axs[1, 0].set_ylabel("S"); axs[1, 0].set_title("(c) S vs \u03b3 by class")
    axs[1, 1].hist(df["gamma"], bins=20, density=True, color="#55A868", alpha=.6, edgecolor="k", lw=.3)
    xs = np.sort(df["gamma"]); ax2 = axs[1, 1].twinx()
    ax2.plot(xs, np.arange(1, len(xs) + 1) / len(xs), "k-", lw=1.5); ax2.set_ylabel("CDF")
    axs[1, 1].set_xlabel("\u03b3"); axs[1, 1].set_title("(d) \u03b3 distribution + CDF")
    fig.suptitle("Figure 6. The compacity ratio \u03b3 as governing variable")
    fig.tight_layout(); fig.savefig(f"{outdir}/Figure06.png", dpi=300); plt.close()

    # ---- Figures 12 & 13 : CV box plots and gamma ablation ----
    if acc is not None:
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.boxplot([acc["CPINN"][o] for o in OUTS], tick_labels=OUTS)
        for i, o in enumerate(OUTS, 1):
            ax.plot([i] * len(acc["CPINN"][o]), acc["CPINN"][o], "k.", ms=8)
        ax.axhline(0, color="r", ls="--", lw=1); ax.set_ylabel("$R^2$ (5-fold CV)")
        ax.set_title("Figure 12. Per-fold $R^2$ \u2014 C-PINN-Att")
        fig.tight_layout(); fig.savefig(f"{outdir}/Figure12.png", dpi=300); plt.close()

        fig, axs = plt.subplots(1, 3, figsize=(11, 3.6))
        for ax, o in zip(axs, OUTS):
            for j in range(len(acc["CPINN"][o])):
                ax.plot([0, 1], [acc["gamma_free"][o][j], acc["CPINN"][o][j]],
                        "o-", color="gray", alpha=.6)
            t, p = stats.ttest_rel(acc["CPINN"][o], acc["gamma_free"][o])
            ax.set_xticks([0, 1]); ax.set_xticklabels(["\u03b3-free", "\u03b3-cond."])
            ax.set_title(f"$R^2$({o})  p={p:.3f}"); ax.set_ylim(0, 1)
        axs[0].set_ylabel("$R^2$")
        fig.suptitle("Figure 13. \u03b3-conditioning ablation (paired, 5 folds)")
        fig.tight_layout(); fig.savefig(f"{outdir}/Figure13.png", dpi=300); plt.close()

    # ---- Figures 16 & 17 : observed vs predicted ----
    f = np.load(os.path.join(outdir, "fixed_split_predictions.npz"), allow_pickle=True)
    pred, obs, gcl = f["pred"], f["obs"], f["gclass"]
    tr = df.index[df["Split"] == "Train"].tolist(); te = df.index[df["Split"] == "Test"].tolist()
    rb = run_baseline(df, tr, te, "dnn")

    fig, axs = plt.subplots(1, 3, figsize=(13, 4))
    for i, ax in enumerate(axs):
        lo, hi = float(obs[:, i].min()), float(obs[:, i].max())
        ax.plot([lo, hi], [lo, hi], "k-", lw=1)
        ax.fill_between([lo, hi], [lo * .95, hi * .95], [lo * 1.05, hi * 1.05], color="tab:blue", alpha=.15)
        ax.fill_between([lo, hi], [lo * .90, hi * .90], [lo * 1.10, hi * 1.10], color="tab:orange", alpha=.10)
        for c in GORDER:
            m = gcl == c
            if m.any():
                ax.scatter(obs[m, i], pred[m, i], color=colors[c], label=c, s=30, edgecolor="k", lw=.3)
        ax.set_xlabel("Observed"); ax.set_ylabel("Predicted")
        ax.set_title(f"{labels[i]}  $R^2$={r2_score(obs[:, i], pred[:, i]):.3f}")
    axs[0].legend(fontsize=7, loc="upper left")
    fig.suptitle("Figure 16. Observed vs predicted \u2014 C-PINN-Att (fixed 114/37 split)")
    fig.tight_layout(); fig.savefig(f"{outdir}/Figure16.png", dpi=300); plt.close()

    fig, axs = plt.subplots(1, 3, figsize=(13, 4))
    for i, ax in enumerate(axs):
        lo, hi = float(obs[:, i].min()), float(obs[:, i].max())
        ax.plot([lo, hi], [lo, hi], "k-", lw=1)
        ax.scatter(obs[:, i], rb["pred"][:, i], color="gray", s=28,
                   label=f"DNN R\u00b2={r2_score(obs[:, i], rb['pred'][:, i]):.3f}")
        ax.scatter(obs[:, i], pred[:, i], color="tab:blue", s=28,
                   label=f"C-PINN-Att R\u00b2={r2_score(obs[:, i], pred[:, i]):.3f}")
        ax.set_xlabel("Observed"); ax.set_ylabel("Predicted"); ax.set_title(labels[i])
        ax.legend(fontsize=7)
    fig.suptitle("Figure 17. DNN vs C-PINN-Att (same test set)")
    fig.tight_layout(); fig.savefig(f"{outdir}/Figure17.png", dpi=300); plt.close()

    # ---- Figure 18 : observed vs predicted with 1:1 AND regression line ----
    fig, axs = plt.subplots(1, 3, figsize=(13, 4))
    for i, ax in enumerate(axs):
        lo, hi = float(obs[:, i].min()), float(obs[:, i].max())
        ax.plot([lo, hi], [lo, hi], "k-", lw=1, label="1:1")
        for c in GORDER:
            m = gcl == c
            if m.any():
                ax.scatter(obs[m, i], pred[m, i], color=colors[c], s=28, edgecolor="k", lw=.3)
        # least-squares regression line pred = a*obs + b
        a, b = np.polyfit(obs[:, i], pred[:, i], 1)
        xs = np.array([lo, hi])
        ax.plot(xs, a * xs + b, "r--", lw=1.5,
                label=f"fit: y={a:.2f}x+{b:.2f}")
        ax.set_xlabel("Observed"); ax.set_ylabel("Predicted")
        ax.set_title(f"{labels[i]}  R\u00b2={r2_score(obs[:, i], pred[:, i]):.3f}")
        ax.legend(fontsize=7)
    fig.suptitle("Figure 18. Observed vs predicted with 1:1 and regression lines (C-PINN-Att)")
    fig.tight_layout(); fig.savefig(f"{outdir}/Figure18.png", dpi=300); plt.close()

    # ---- Figure 19 : residuals vs predicted ----
    resid = obs - pred
    fig, axs = plt.subplots(1, 3, figsize=(13, 4))
    for i, ax in enumerate(axs):
        ax.scatter(pred[:, i], resid[:, i], color="tab:purple", s=26, edgecolor="k", lw=.3)
        ax.axhline(0, color="k", lw=1)
        ax.set_xlabel("Predicted"); ax.set_ylabel("Residual (obs \u2212 pred)")
        ax.set_title(labels[i])
    fig.suptitle("Figure 19. Residuals vs predicted (fixed split)")
    fig.tight_layout(); fig.savefig(f"{outdir}/Figure19.png", dpi=300); plt.close()

    # ---- Figure 20 : histogram of residuals ----
    fig, axs = plt.subplots(1, 3, figsize=(13, 4))
    for i, ax in enumerate(axs):
        ax.hist(resid[:, i], bins=15, color="#55A868", alpha=.75, edgecolor="k", lw=.3, density=True)
        mu, sd = resid[:, i].mean(), resid[:, i].std() + 1e-9
        xs = np.linspace(resid[:, i].min(), resid[:, i].max(), 200)
        ax.plot(xs, np.exp(-0.5 * ((xs - mu) / sd) ** 2) / (sd * np.sqrt(2 * np.pi)),
                "k-", lw=1.2)
        ax.axvline(0, color="r", ls="--", lw=1)
        ax.set_xlabel("Residual"); ax.set_ylabel("density"); ax.set_title(labels[i])
    fig.suptitle("Figure 20. Histogram of residuals with Gaussian reference (fixed split)")
    fig.tight_layout(); fig.savefig(f"{outdir}/Figure20.png", dpi=300); plt.close()

    # ---- Figure 21 : training-loss curves (total / data / physics) ----
    hist = R.get("loss_history")
    if hist and hist.get("epoch"):
        fig, ax = plt.subplots(figsize=(7, 4.5))
        ax.plot(hist["epoch"], hist["total"], label="Total loss", lw=1.6)
        ax.plot(hist["epoch"], hist["data"], label="Data loss", lw=1.3)
        ax.plot(hist["epoch"], hist["physics"], label="Physics loss", lw=1.3)
        ax.set_yscale("log")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss (log scale)")
        ax.set_title("Figure 21. Training-loss evolution (C-PINN-Att, fixed split)")
        ax.legend(); fig.tight_layout()
        fig.savefig(f"{outdir}/Figure21.png", dpi=300); plt.close()

    # ---- bump the earlier data/result figures to 300 dpi as well ----
    #      (re-save the ones created above at 200 dpi is unnecessary here;
    #       new figures 17-21 are already at 300 dpi.)
    print(f"\nFigures written to {outdir}/ (300 dpi for results/diagnostics)")


# -----------------------------------------------------------------------------
# 21. RESULT EXPORTS
# Source: cpinn_att_reproduce.py
# -----------------------------------------------------------------------------
# =============================================================================
# 8.  MAIN
# =============================================================================
def export_results_excel(R: dict, script_dir: str,
                         filename: str = "CPINN_Att_Results.xlsx") -> str:
    """Write every computed result into a multi-sheet Excel workbook.

    The file is saved in `script_dir` (the folder that contains this script),
    next to the code, as requested. Each manuscript item becomes one sheet.
    Returns the full path of the written file.
    """
    path = os.path.join(script_dir, filename)
    frames: dict[str, pd.DataFrame] = {}

    # -- Config / run metadata -------------------------------------------------
    cfg = R.get("config", {})
    frames["Config"] = pd.DataFrame(
        [{"parameter": k, "value": (json.dumps(v) if isinstance(v, (dict, list)) else v)}
         for k, v in cfg.items()])

    # -- Table 1 : descriptive statistics -------------------------------------
    if "table1" in R:
        frames["Table1_Statistics"] = pd.DataFrame(R["table1"])

    # -- Table 4 : 5-fold CV (mean ± std) -------------------------------------
    if "table4" in R:
        rows = []
        for model, d in R["table4"].items():
            rows.append({
                "Model": model,
                "R2(Y)": f"{d['Y'][0]:.3f} ± {d['Y'][1]:.3f}",
                "R2(S)": f"{d['S'][0]:.3f} ± {d['S'][1]:.3f}",
                "R2(X)": f"{d['X'][0]:.3f} ± {d['X'][1]:.3f}",
                "PVI(%)": f"{d['PVI'][0]:.1f} ± {d['PVI'][1]:.1f}"})
        frames["Table4_CV"] = pd.DataFrame(rows)

    # -- Table 4 raw per-fold values ------------------------------------------
    if "table4_raw" in R:
        rows = []
        for model, d in R["table4_raw"].items():
            nf = len(next(iter(d.values())))
            for f in range(nf):
                rows.append({"Model": model, "Fold": f + 1,
                             **{o: round(d[o][f], 4) for o in d}})
        frames["Table4_perFold"] = pd.DataFrame(rows)

    # -- t-tests --------------------------------------------------------------
    def ttest_frame(key, label):
        if key not in R:
            return
        rows = [{"Output": o, "t": round(v[0], 3), "p": v[1], "significance": v[2]}
                for o, v in R[key].items()]
        frames[label] = pd.DataFrame(rows)
    ttest_frame("ttest_gamma_free", "Ttest_gamma_free")
    ttest_frame("ttest_dnn", "Ttest_vs_DNN")

    # -- Table 5 : per-gamma-class --------------------------------------------
    if "table5" in R:
        rows = []
        for c, d in R["table5"].items():
            rows.append({"gamma_class": c, "mean_Ntest": round(d.get("n", 0), 1),
                         **{o: (round(d[o], 3) if d.get(o) is not None else None)
                            for o in OUTS}})
        frames["Table5_perGammaClass"] = pd.DataFrame(rows)

    # -- Table 6 : ablation ----------------------------------------------------
    if "table6" in R:
        rows = [{"Stage": s, "R2_mean": round(v[0], 3),
                 "RMSE(%)": round(v[1], 2), "PVI(%)": round(v[2], 1)}
                for s, v in R["table6"].items()]
        frames["Table6_Ablation"] = pd.DataFrame(rows)

    # -- Table 7 : benchmark ---------------------------------------------------
    if "table7" in R:
        rows = []
        for model, d in R["table7"].items():
            rows.append({"Model": model,
                         **{f"R2({o})": round(d["R2"][i], 3) for i, o in enumerate(OUTS)},
                         "RMSE(%)": round(d["RMSE"], 2), "PVI(%)": round(d["PVI"], 1)})
        frames["Table7_Benchmark"] = pd.DataFrame(rows)

    # -- Extended global metrics on the fixed split (R2/RMSE/MAE/MAPE/NSE) ----
    if "fixed_metrics" in R:
        rows = []
        for o, d in R["fixed_metrics"].items():
            rows.append({"Output": o,
                         "R2": round(d["R2"], 4), "RMSE": round(d["RMSE"], 4),
                         "MAE": round(d["MAE"], 4), "MAPE(%)": round(d["MAPE"], 3),
                         "NSE": round(d["NSE"], 4)})
        frames["Metrics_FixedSplit"] = pd.DataFrame(rows)

    # -- Per-sample observed / predicted / abs & rel error --------------------
    if "obs_pred_table" in R:
        frames["Observed_vs_Predicted"] = pd.DataFrame(R["obs_pred_table"])

    # -- Training loss history ------------------------------------------------
    if "loss_history" in R and R["loss_history"].get("epoch"):
        frames["Loss_History"] = pd.DataFrame(R["loss_history"])

    # -- Table 8 : fixed split + bootstrap CI ---------------------------------
    if "table8" in R:
        rows = []
        for o, d in R["table8"].items():
            rows.append({"Output": o, "R2": round(d["R2"], 3),
                         "CI_low": round(d["CI"][0], 3), "CI_high": round(d["CI"][1], 3),
                         "RMSE": round(d["RMSE"], 3), "NRMSE(%)": round(d["NRMSE_pct"], 2),
                         "MAE": round(d["MAE"], 3)})
        frames["Table8_FixedSplit_CI"] = pd.DataFrame(rows)

    # -- Multi-seed robustness -------------------------------------------------
    if "multiseed_performance" in R:
        mp = R["multiseed_performance"]
        rows = []
        for i, sd in enumerate(mp["seeds"]):
            rows.append({"seed": sd,
                         **{f"R2({o})": round(mp["raw"][o][i], 4) for o in OUTS},
                         "PVI(%)": round(mp["raw"]["PVI"][i], 2)})
        rows.append({"seed": "mean +- std",
                     **{f"R2({o})": f"{mp['summary'][o][0]:.3f} +- {mp['summary'][o][1]:.3f}"
                        for o in OUTS},
                     "PVI(%)": f"{mp['summary']['PVI'][0]:.1f} +- {mp['summary']['PVI'][1]:.1f}"})
        frames["MultiSeed_Performance"] = pd.DataFrame(rows)
    if "multiseed_diagnostic" in R:
        frames["MultiSeed_GradientPaths"] = pd.DataFrame(R["multiseed_diagnostic"]["rows"])

    # -- Section 3.8 diagnostics ----------------------------------------------
    diag_rows = []
    if "shapiro" in R:
        for o, v in R["shapiro"].items():
            diag_rows.append({"diagnostic": f"Shapiro-Wilk {o}", "value": round(v[0], 3),
                              "p_value": v[1]})
    if "calibration" in R:
        for o, v in R["calibration"].items():
            diag_rows.append({"diagnostic": f"MC-dropout coverage {o} (±1σ,%)",
                              "value": round(v[0], 1), "p_value": ""})
            diag_rows.append({"diagnostic": f"MC-dropout coverage {o} (±2σ,%)",
                              "value": round(v[1], 1), "p_value": ""})
    if "fixed_PVI" in R:
        diag_rows.append({"diagnostic": "PVI fixed split (%)",
                          "value": round(R["fixed_PVI"], 1), "p_value": ""})
    if diag_rows:
        frames["Sec38_Diagnostics"] = pd.DataFrame(diag_rows)

    # -- Functional consistency (C-1..C-4) ------------------------------------
    if "functional_consistency" in R:
        fc = R["functional_consistency"]
        frames["Sec38_FunctionalChecks"] = pd.DataFrame([
            {"check": "C-1 Belanger mean deviation (%)",
             "value": round(fc["belanger_mean_dev_pct"], 2)},
            {"check": "C-1 Belanger Spearman rho",
             "value": round(fc["belanger_spearman_rho"], 3)},
            {"check": "C-1 Belanger Spearman p",
             "value": fc["belanger_spearman_p"]},
            {"check": "C-4 Forster-Skrinde R2 vs observed",
             "value": round(fc["fs_R2_observed"], 3)},
            {"check": "C-4 Forster-Skrinde R2 vs predicted",
             "value": round(fc["fs_R2_predicted"], 3)},
        ])

    # -- UQ : ensemble + MC-Dropout calibration --------------------------------
    if "uq_ensemble_mc_dropout" in R:
        cal = R["uq_ensemble_mc_dropout"]["calibration"]
        rows = [{"Output": o, "Mean_predictive_std": round(cal[o]["mean_std"], 4),
                 "Coverage_1sigma_pct": round(cal[o]["coverage_1sigma_pct"], 1),
                 "Coverage_2sigma_pct": round(cal[o]["coverage_2sigma_pct"], 1),
                 "Nominal_1sigma_pct": 68.3, "Nominal_2sigma_pct": 95.4}
                for o in OUTS]
        frames["UQ_Calibration"] = pd.DataFrame(rows)

    # -- Applicability domain : relative error inside/outside convex hull -----
    if "applicability_domain_fixed_split" in R:
        ad = R["applicability_domain_fixed_split"]
        rows = []
        for o in OUTS:
            d = ad[o]
            rows.append({"Output": o, "Domain": "inside_hull", "n": d["n_inside"],
                         "Mean_rel_err_%": round(d["mean_in"], 3) if d["mean_in"] is not None else None,
                         "Std_rel_err_%": round(d["std_in"], 3) if d["std_in"] is not None else None,
                         "Max_rel_err_%": round(d["max_in"], 3) if d["max_in"] is not None else None})
            rows.append({"Output": o, "Domain": "outside_hull", "n": d["n_outside"],
                         "Mean_rel_err_%": round(d["mean_out"], 3) if d["mean_out"] is not None else None,
                         "Std_rel_err_%": round(d["std_out"], 3) if d["std_out"] is not None else None,
                         "Max_rel_err_%": round(d["max_out"], 3) if d["max_out"] is not None else None})
        frames["Applicability_Domain"] = pd.DataFrame(rows)

    # -- Write workbook --------------------------------------------------------
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        for sheet, frame in frames.items():
            frame.to_excel(xw, sheet_name=sheet[:31], index=False)

    # -- Mirror every sheet to CSV (same folder, in a 'csv' subdirectory) -----
    csv_dir = os.path.join(script_dir, "CPINN_Att_Results_csv")
    os.makedirs(csv_dir, exist_ok=True)
    for sheet, frame in frames.items():
        frame.to_csv(os.path.join(csv_dir, f"{sheet}.csv"), index=False)

    return path

# -----------------------------------------------------------------------------
# 24. UNCERTAINTY QUANTIFICATION  (Deep Ensemble + MC-Dropout, combined)
# -----------------------------------------------------------------------------
# =============================================================================
# 9.  UQ : DEEP ENSEMBLE + MC-DROPOUT
# =============================================================================
def _to_csv(frame: pd.DataFrame, outdir: str, filename: str) -> str:
    """Small IO helper: every CSV in Sections 24/25 goes through here so the
    'written to disk' behaviour and console message are consistent."""
    csv_dir = os.path.join(outdir, "csv")
    os.makedirs(csv_dir, exist_ok=True)
    path = os.path.join(csv_dir, filename)
    frame.to_csv(path, index=False)
    print(f"  [csv] {path}  ({len(frame)} rows)")
    return path


def run_ensemble_mc_uq(df, tr, te, ep, n_ensemble=5, mc_T=200, seeds=SEEDS, verbose=True):
    """Deep-ensemble + MC-Dropout predictive uncertainty on the fixed test set.

    For each of `n_ensemble` independently-initialised models (different
    seeds, same architecture / lambdas / protocol as the reported model):
      1. train to convergence exactly as in run_cpinn();
      2. at inference, re-enable Dropout only ("MC-Dropout") and draw
         `mc_T` stochastic forward passes -> per-model mean and variance.

    The two variance sources are combined through the law of total variance

        Var_total = E_model[Var_dropout]  +  Var_model[E_dropout]
                    \\________________/     \\________________/
                     within-model term        between-model term
                     (weight-sampling         (epistemic model
                      / MC-Dropout)            disagreement)

    i.e. a deep ensemble and MC-Dropout are fused rather than one method
    being picked and the other discarded (Lakshminarayanan et al. 2017;
    Gal & Ghahramani 2016).
    """
    seeds = list(seeds)[:n_ensemble]
    all_means, all_vars, ensemble_raw = [], [], []
    ref = None
    for k, sd in enumerate(seeds):
        r = run_cpinn(df, tr, te, ep, seed=sd, return_model=True)
        if ref is None:
            ref = r  # keep one run for obs / test_rows (identical across seeds)
        model, Xte_n = r["model"], r["Xte_n"]
        _, _, mu_y, sd_y = r["norm"]
        model.eval()
        for m in model.modules():                 # MC-Dropout: dropout only
            if isinstance(m, nn.Dropout):
                m.train()
        samp = np.zeros((mc_T, len(te), 3))
        with torch.no_grad():
            for t in range(mc_T):
                Yh, Sh, Xh, _ = model(Xte_n)
                samp[t] = (torch.cat([Yh, Sh, Xh], -1).numpy() * sd_y + mu_y)
        member_mean, member_var = samp.mean(0), samp.var(0)
        all_means.append(member_mean)
        all_vars.append(member_var)
        for j in range(len(te)):
            row = {"ensemble_member": k, "seed": sd, "sample": j + 1}
            for i, o in enumerate(OUTS):
                row[f"{o}_mc_mean"] = float(member_mean[j, i])
                row[f"{o}_mc_std"] = float(np.sqrt(member_var[j, i] + 1e-12))
            ensemble_raw.append(row)
        if verbose:
            print(f"  [UQ] ensemble member {k + 1}/{n_ensemble} (seed={sd}) done")

    all_means = np.array(all_means)                  # (M, N, 3)
    all_vars = np.array(all_vars)                     # (M, N, 3)
    pred_mean = all_means.mean(0)                      # (N, 3) final point estimate
    var_ensemble = all_means.var(0)                     # model-to-model disagreement
    var_dropout = all_vars.mean(0)                      # avg. within-model dropout var
    pred_std = np.sqrt(var_ensemble + var_dropout + 1e-12)

    obs = ref["obs"]
    cov = {}
    for i, o in enumerate(OUTS):
        z = np.abs(obs[:, i] - pred_mean[:, i]) / (pred_std[:, i] + 1e-12)
        cov[o] = {"coverage_1sigma_pct": float(100 * np.mean(z <= 1)),
                  "coverage_2sigma_pct": float(100 * np.mean(z <= 2)),
                  "mean_std": float(pred_std[:, i].mean())}
    if verbose:
        print("\n=== Ensemble + MC-Dropout calibration (combined std) ===")
        for o in OUTS:
            print(f"  {o}: mean_sigma={cov[o]['mean_std']:.4f}  "
                  f"cov(+-1s)={cov[o]['coverage_1sigma_pct']:.1f}%  "
                  f"cov(+-2s)={cov[o]['coverage_2sigma_pct']:.1f}%  (nominal 68.3/95.4)")

    return dict(pred_mean=pred_mean, pred_std=pred_std,
               ensemble_std=np.sqrt(var_ensemble + 1e-12),
               dropout_std=np.sqrt(var_dropout + 1e-12),
               obs=obs, test_rows=ref["test_rows"].reset_index(drop=True),
               calibration=cov, n_ensemble=n_ensemble, mc_T=mc_T,
               ensemble_raw=ensemble_raw, seeds_used=seeds)


def export_uq_predictions_csv(uq, ad, rel_stats, outdir):
    """Per-sample table: observed, predictive mean/std, error, hull flag."""
    test_rows = uq["test_rows"]
    n = len(uq["obs"])
    rows = []
    for j in range(n):
        row = {"sample": j + 1,
               "gamma_class": test_rows["gamma_class"].iloc[j] if "gamma_class" in test_rows else None,
               "Fr1": float(test_rows["Fr1"].iloc[j]) if "Fr1" in test_rows else None,
               "inside_convex_hull": bool(ad["inside_mask"][j])}
        for i, o in enumerate(OUTS):
            row[f"{o}_obs"] = float(uq["obs"][j, i])
            row[f"{o}_pred_mean"] = float(uq["pred_mean"][j, i])
            row[f"{o}_pred_std_total"] = float(uq["pred_std"][j, i])
            row[f"{o}_ensemble_std"] = float(uq["ensemble_std"][j, i])
            row[f"{o}_dropout_std"] = float(uq["dropout_std"][j, i])
            row[f"{o}_rel_err_pct"] = float(rel_stats[o]["rel_err_all"][j])
        rows.append(row)
    return _to_csv(pd.DataFrame(rows), outdir, "uq_predictions_per_sample.csv")


def export_ensemble_raw_csv(uq, outdir):
    """Long-format table: one row per (ensemble member x test sample)."""
    return _to_csv(pd.DataFrame(uq["ensemble_raw"]), outdir, "uq_ensemble_raw_predictions.csv")


def export_calibration_csv(uq, outdir):
    rows = [{"Output": o, "mean_predictive_std": uq["calibration"][o]["mean_std"],
             "coverage_1sigma_pct": uq["calibration"][o]["coverage_1sigma_pct"],
             "coverage_2sigma_pct": uq["calibration"][o]["coverage_2sigma_pct"],
             "nominal_1sigma_pct": 68.3, "nominal_2sigma_pct": 95.4,
             "n_ensemble": uq["n_ensemble"], "mc_T": uq["mc_T"]} for o in OUTS]
    return _to_csv(pd.DataFrame(rows), outdir, "uq_calibration_metrics.csv")


# -----------------------------------------------------------------------------
# 25. APPLICABILITY DOMAIN  (PCA(2) convex hull) + error-vs-domain diagnostics
# -----------------------------------------------------------------------------
# =============================================================================
# 10. APPLICABILITY DOMAIN : PCA(2) CONVEX HULL
# =============================================================================
def applicability_domain_pca(df, tr, te, seed=SEED):
    """2-D PCA convex hull of the standardized training features.

    The hull is built from TRAIN points only, in a PCA space fitted on TRAIN
    only (no leakage). Test points are then classified as inside (true
    interpolation w.r.t. the training input distribution) or outside
    (extrapolation) via a Delaunay triangulation of the hull vertices --
    the numerically robust way to do a point-in-hull test in 2D.
    """
    Xtr = df.loc[tr, FEATS].values.astype(float)
    Xte = df.loc[te, FEATS].values.astype(float)
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-8
    Xtr_n, Xte_n = (Xtr - mu) / sd, (Xte - mu) / sd

    pca = PCA(n_components=2, random_state=seed)
    Ztr = pca.fit_transform(Xtr_n)
    Zte = pca.transform(Xte_n)

    hull = ConvexHull(Ztr)
    tri = Delaunay(Ztr[hull.vertices])
    inside = tri.find_simplex(Zte) >= 0

    return dict(pca=pca, Ztr=Ztr, Zte=Zte, hull=hull, inside_mask=inside,
               explained_variance_ratio=pca.explained_variance_ratio_)


def relative_error_by_domain(pred, obs, inside_mask, R=None, label="fixed_split"):
    """Per-output relative-error (%) statistics split by hull membership."""
    stats_out = {}
    for i, o in enumerate(OUTS):
        rel = 100 * np.abs(obs[:, i] - pred[:, i]) / (np.abs(obs[:, i]) + 1e-12)
        rin, rout = rel[inside_mask], rel[~inside_mask]
        stats_out[o] = dict(
            n_inside=int(inside_mask.sum()), n_outside=int((~inside_mask).sum()),
            mean_in=float(rin.mean()) if rin.size else None,
            std_in=float(rin.std()) if rin.size else None,
            max_in=float(rin.max()) if rin.size else None,
            mean_out=float(rout.mean()) if rout.size else None,
            std_out=float(rout.std()) if rout.size else None,
            max_out=float(rout.max()) if rout.size else None,
            rel_err_all=rel)
        print(f"  {o}:  inside  (n={stats_out[o]['n_inside']:3d})  "
              f"mean={stats_out[o]['mean_in']:.2f}%  std={stats_out[o]['std_in']:.2f}%  "
              f"max={stats_out[o]['max_in']:.2f}%")
        if rout.size:
            print(f"       outside (n={stats_out[o]['n_outside']:3d})  "
                  f"mean={stats_out[o]['mean_out']:.2f}%  std={stats_out[o]['std_out']:.2f}%  "
                  f"max={stats_out[o]['max_out']:.2f}%")
        else:
            print("       outside: no test points fall outside the training convex hull")
    if R is not None:
        R[f"applicability_domain_{label}"] = {
            o: {k: v for k, v in d.items() if k != "rel_err_all"}
            for o, d in stats_out.items()}
    return stats_out


def export_applicability_domain_csv(ad, outdir):
    """Train and test points in PCA(2) space, with hull-vertex / inside flags."""
    Ztr, Zte = ad["Ztr"], ad["Zte"]
    hull_vertex_idx = set(ad["hull"].vertices.tolist())
    rows = []
    for j in range(len(Ztr)):
        rows.append({"split": "train", "sample": j + 1, "PC1": float(Ztr[j, 0]),
                     "PC2": float(Ztr[j, 1]), "is_hull_vertex": j in hull_vertex_idx,
                     "inside_convex_hull": True})
    for j in range(len(Zte)):
        rows.append({"split": "test", "sample": j + 1, "PC1": float(Zte[j, 0]),
                     "PC2": float(Zte[j, 1]), "is_hull_vertex": False,
                     "inside_convex_hull": bool(ad["inside_mask"][j])})
    _to_csv(pd.DataFrame(rows), outdir, "applicability_domain_pca_points.csv")

    hv = ad["hull"].vertices
    hull_frame = pd.DataFrame({"vertex_order": range(len(hv)),
                               "train_sample": hv + 1,
                               "PC1": ad["Ztr"][hv, 0], "PC2": ad["Ztr"][hv, 1]})
    _to_csv(hull_frame, outdir, "convex_hull_vertices.csv")

    evr = pd.DataFrame({"component": ["PC1", "PC2"],
                        "explained_variance_ratio": ad["explained_variance_ratio"]})
    return _to_csv(evr, outdir, "pca_explained_variance.csv")


def export_relative_error_stats_csv(rel_stats, outdir):
    rows = []
    for o in OUTS:
        d = rel_stats[o]
        rows.append({"Output": o, "domain": "inside_hull", "n": d["n_inside"],
                     "mean_rel_err_pct": d["mean_in"], "std_rel_err_pct": d["std_in"],
                     "max_rel_err_pct": d["max_in"]})
        rows.append({"Output": o, "domain": "outside_hull", "n": d["n_outside"],
                     "mean_rel_err_pct": d["mean_out"], "std_rel_err_pct": d["std_out"],
                     "max_rel_err_pct": d["max_out"]})
    return _to_csv(pd.DataFrame(rows), outdir, "relative_error_by_domain.csv")


def make_uq_and_domain_figures(uq, ad, rel_stats, outdir):
    """Figure 22 (UQ intervals) and Figure 23a/b (PCA hull + error histograms).

    All figures are written as 300-dpi PNG under <outdir>/figures/.
    """
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig_dir = os.path.join(outdir, "figures")
    os.makedirs(fig_dir, exist_ok=True)

    lab_map = {"Y": "Y = h\u2082/h\u2081", "S": "S = s/h\u2081", "X": "X = x/h\u2081"}
    lab = [lab_map.get(o, o) for o in OUTS]
    obs, pred, sd = uq["obs"], uq["pred_mean"], uq["pred_std"]
    saved = []

    # ---- Figure 22 : predictive interval coverage (ensemble + MC-Dropout) --
    fig, axs = plt.subplots(1, 3, figsize=(13, 4))
    for i, ax in enumerate(axs):
        idx = np.argsort(obs[:, i])
        x = np.arange(len(idx))
        ax.fill_between(x, pred[idx, i] - 2 * sd[idx, i], pred[idx, i] + 2 * sd[idx, i],
                        color="tab:blue", alpha=.15, label="\u00b12\u03c3")
        ax.fill_between(x, pred[idx, i] - sd[idx, i], pred[idx, i] + sd[idx, i],
                        color="tab:blue", alpha=.30, label="\u00b11\u03c3")
        ax.plot(x, pred[idx, i], "b-", lw=1.2, label="predictive mean")
        ax.plot(x, obs[idx, i], "ko", ms=3, label="observed")
        ax.set_title(f"{lab[i]}  "
                     f"cov1\u03c3={uq['calibration'][OUTS[i]]['coverage_1sigma_pct']:.0f}%  "
                     f"cov2\u03c3={uq['calibration'][OUTS[i]]['coverage_2sigma_pct']:.0f}%")
        ax.set_xlabel("test sample (sorted by observed value)")
        if i == 0:
            ax.legend(fontsize=7)
    fig.suptitle(f"Figure 22. Ensemble (n={uq['n_ensemble']}) + MC-Dropout (T={uq['mc_T']}) "
                "predictive intervals, fixed test split")
    fig.tight_layout()
    p = os.path.join(fig_dir, "Figure22_UQ_intervals.png")
    fig.savefig(p, dpi=300); plt.close(); saved.append(p)

    # ---- Figure 23a : PCA(2) convex hull + inside/outside classification ---
    Ztr, Zte, hull, inside = ad["Ztr"], ad["Zte"], ad["hull"], ad["inside_mask"]
    fig, ax = plt.subplots(figsize=(6, 5.5))
    ax.scatter(Ztr[:, 0], Ztr[:, 1], s=22, color="0.6", label="train", alpha=.7)
    for simplex in hull.simplices:
        ax.plot(Ztr[simplex, 0], Ztr[simplex, 1], "k-", lw=1)
    ax.scatter(Zte[inside, 0], Zte[inside, 1], s=40, color="tab:green",
               edgecolor="k", lw=.4, label="test \u2014 inside hull")
    if (~inside).any():
        ax.scatter(Zte[~inside, 0], Zte[~inside, 1], s=55, color="tab:red", marker="^",
                   edgecolor="k", lw=.4, label="test \u2014 outside hull (extrapolation)")
    ax.set_xlabel(f"PC1 ({100 * ad['explained_variance_ratio'][0]:.1f}% var)")
    ax.set_ylabel(f"PC2 ({100 * ad['explained_variance_ratio'][1]:.1f}% var)")
    ax.set_title("Figure 23a. Applicability domain \u2014 PCA(2) convex hull of "
                 "(Fr\u2081, h\u2081/\u0125\u2081, \u03b3)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    p = os.path.join(fig_dir, "Figure23a_convex_hull.png")
    fig.savefig(p, dpi=300); plt.close(); saved.append(p)

    # ---- Figure 23b : histogram of relative error, inside vs outside -------
    fig, axs = plt.subplots(1, 3, figsize=(13, 4))
    for i, (ax, o) in enumerate(zip(axs, OUTS)):
        rel = rel_stats[o]["rel_err_all"]
        rin, rout = rel[inside], rel[~inside]
        bins = np.linspace(0, max(rel.max(), 1e-6), 20)
        ax.hist(rin, bins=bins, color="tab:green", alpha=.6, edgecolor="k", lw=.3,
                label=f"inside (n={rin.size})", density=True)
        if rout.size:
            ax.hist(rout, bins=bins, color="tab:red", alpha=.6, edgecolor="k", lw=.3,
                    label=f"outside (n={rout.size})", density=True)
        ax.axvline(rin.mean(), color="tab:green", ls="--", lw=1.4)
        if rout.size:
            ax.axvline(rout.mean(), color="tab:red", ls="--", lw=1.4)
        ax.set_xlabel("relative error (%)"); ax.set_ylabel("density")
        ax.set_title(lab[i]); ax.legend(fontsize=7)
    fig.suptitle("Figure 23b. Relative-error distribution inside vs outside "
                "the training convex hull")
    fig.tight_layout()
    p = os.path.join(fig_dir, "Figure23b_error_hist_by_domain.png")
    fig.savefig(p, dpi=300); plt.close(); saved.append(p)

    print(f"\n[png] {len(saved)} figures written to {fig_dir}/")
    return saved


def run_uq_and_applicability_domain(df, tr, te, ep, R, outdir, n_ensemble=5, mc_T=200):
    """Driver for Sections 24-25. Trains the ensemble, builds the applicability
    domain, computes every metric, and writes ALL data / predictions / metrics
    to <outdir>/csv/*.csv and ALL plots to <outdir>/figures/*.png.
    """
    print("\n" + "=" * 74)
    print(" UQ (deep ensemble + MC-Dropout) and applicability-domain diagnostics")
    print("=" * 74)
    os.makedirs(os.path.join(outdir, "csv"), exist_ok=True)
    os.makedirs(os.path.join(outdir, "figures"), exist_ok=True)

    # ---- 1. uncertainty quantification --------------------------------------
    uq = run_ensemble_mc_uq(df, tr, te, ep, n_ensemble=n_ensemble, mc_T=mc_T)
    R["uq_ensemble_mc_dropout"] = {
        "n_ensemble": uq["n_ensemble"], "mc_T": uq["mc_T"],
        "calibration": uq["calibration"]}

    # ---- 2. applicability domain ---------------------------------------------
    ad = applicability_domain_pca(df, tr, te)
    print(f"\n=== PCA(2) applicability domain ===  "
          f"explained variance = {100 * ad['explained_variance_ratio'].sum():.1f}% "
          f"({100 * ad['explained_variance_ratio'][0]:.1f}% + "
          f"{100 * ad['explained_variance_ratio'][1]:.1f}%)")
    print(f"  {int(ad['inside_mask'].sum())}/{len(ad['inside_mask'])} test points fall "
          f"inside the training convex hull "
          f"({int((~ad['inside_mask']).sum())} extrapolated)")

    # ---- 3. relative error split by domain -----------------------------------
    print("\n=== Relative error (%) \u2014 inside vs outside the convex hull ===")
    rel_stats = relative_error_by_domain(uq["pred_mean"], uq["obs"], ad["inside_mask"], R=R)

    # ---- 4. CSV exports (ALL data / predictions / metrics) -------------------
    print("\n=== Writing CSV outputs ===")
    csv_paths = {
        "uq_predictions_per_sample": export_uq_predictions_csv(uq, ad, rel_stats, outdir),
        "uq_ensemble_raw": export_ensemble_raw_csv(uq, outdir),
        "uq_calibration": export_calibration_csv(uq, outdir),
        "relative_error_by_domain": export_relative_error_stats_csv(rel_stats, outdir),
    }
    csv_paths["applicability_domain_pca_points"] = export_applicability_domain_csv(ad, outdir)

    # ---- 5. PNG figures --------------------------------------------------------
    print("\n=== Writing PNG figures ===")
    fig_paths = make_uq_and_domain_figures(uq, ad, rel_stats, outdir)

    R["uq_applicability_domain_files"] = {"csv": csv_paths, "png": fig_paths}
    return uq, ad, rel_stats, csv_paths, fig_paths


# =============================================================================
# 26. MAIN EXECUTION SECTION
#     Mirrors the execution order of cpinn_att_reproduce.py main(), driven by
#     the RUN_* switches in the USER CONFIGURATION section above, plus the
#     additive UQ / applicability-domain block (Sections 24-25).
# =============================================================================
df = load_dataframe(DATA_FILE)
folds = folds_of(df)
tr = df.index[df["Split"] == "Train"].tolist()
te = df.index[df["Split"] == "Test"].tolist()

R: dict = {"config": dict(seed=SEED, epochs=N_EPOCHS, gamma_th=GAMMA_TH,
                          lambdas=LAMBDAS_PAPER, n=len(df),
                          n_train=len(tr), n_test=len(te))}

print("=" * 74)
print(f" C-PINN-ATT reproduction  |  N = {len(df)}  |  split {len(tr)}/{len(te)}  "
      f"|  epochs = {N_EPOCHS}  |  seed = {SEED}")
print("=" * 74)

# --- Table 1 : descriptive statistics ---------------------------------------
print("\n=== Table 1 : descriptive statistics ===")
raw_h1 = None
try:                                    # optional raw sheet for h1 in cm
    raw = pd.ExcelFile(DATA_FILE).parse("Raw_Data")
    col = [c for c in raw.columns if str(c).lower().startswith("h1")]
    if col:
        raw_h1 = raw[col[0]].values.astype(float)
except Exception:
    pass
t1 = table1(df, raw_h1)
print(t1.to_string(index=False))
R["table1"] = t1.to_dict(orient="records")

# --- Tables 4, 5, 6 and Section 3.8 -----------------------------------------
acc = None
if RUN_TABLE4 or RUN_FIGURES:
    acc = exp_table4(df, folds, N_EPOCHS, R)
if RUN_TABLE5:
    exp_table5(df, folds, N_EPOCHS, R)
if RUN_TABLE6:
    exp_table6(df, tr, te, N_EPOCHS, R)
if RUN_TABLE8 or RUN_FIGURES:
    exp_section38(df, tr, te, N_EPOCHS, R, OUTPUT_DIR)

# --- UQ (deep ensemble + MC-Dropout) + applicability domain (PCA(2) hull) ---
if RUN_UQ:
    uq_result, ad_result, rel_err_result, uq_csv_paths, uq_fig_paths = \
        run_uq_and_applicability_domain(df, tr, te, N_EPOCHS, R, OUTPUT_DIR,
                                        n_ensemble=UQ_N_ENSEMBLE, mc_T=UQ_MC_T)

# --- Final five-seed benchmark (ANFIS / XGBoost / RF / MLP / C-PINN-ATT) -----
# Source: benchmark_models.py
if RUN_BENCHMARK:
    bench_rows = run_benchmark(DATA_FILE, seeds=SEEDS)
    R["table7_final_benchmark"] = bench_rows

# --- Exports ----------------------------------------------------------------
with open(os.path.join(OUTPUT_DIR, "results.json"), "w") as fh:
    json.dump(R, fh, indent=2, default=float)
print(f"\nMetrics written to {OUTPUT_DIR}/results.json")

xlsx_out = export_results_excel(R, os.getcwd())
print(f"Results workbook written to {xlsx_out}")

# --- Figures ----------------------------------------------------------------
if RUN_FIGURES:
    make_figures(df, acc, R, OUTPUT_DIR, N_EPOCHS)

print("\nDone.")


  [check] all inputs and outputs are dimensionless and O(1)-scaled
 C-PINN-ATT reproduction  |  N = 151  |  split 114/37  |  epochs = 2000  |  seed = 42

=== Table 1 : descriptive statistics ===
Variable   Role   Min    Max   Mean  Sigma  CV_pct  Skew
     Fr1  Input 2.800  9.890  6.027  1.778  29.496 0.224
   h1_cm  Input 1.500  4.500  2.523  0.674  26.736 0.994
   gamma  Input 0.893  2.217  1.310  0.308  23.526 0.999
       Y Output 3.357 12.688  7.432  2.473  33.275 0.311
       S Output 1.071  6.250  3.026  1.293  42.739 0.603
       X Output 8.393 56.522 27.247 11.878  43.593 0.649

=== Table 4 : stratified 5-fold CV ===
  DNN         R2(Y)=0.985±0.005  R2(S)=0.978±0.006  R2(X)=0.976±0.012  PVI=1.3±1.6
  gamma_free  R2(Y)=0.951±0.014  R2(S)=0.854±0.050  R2(X)=0.699±0.097  PVI=2.6±3.9
  CPINN       R2(Y)=0.982±0.004  R2(S)=0.965±0.005  R2(X)=0.974±0.007  PVI=1.3±1.6

=== Section 3.2 : gamma-conditioning ablation (paired t-test) ===
  R2(Y): full=0.982 free=0.951  t=6.23 p=0.0034 **